In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision.transforms import v2
import numpy as np
from torchsummary import summary

In [ ]:
# def frequency_based_relu(x, t=1.0, dims=(-2, -1)):
#     t_tensor = torch.as_tensor(t, device=x.device, dtype=x.real.dtype)

#     freq = torch.fft.fftn(x, dim=dims)

#     freq_relu = torch.complex(
#         torch.maximum(freq.real, t_tensor),
#         torch.maximum(freq.imag, t_tensor),
#     )

#     pix = torch.fft.ifftn(freq_relu, dim=dims)
#     return pix.real

# class fReLU(nn.Module):
#     def __init__(self, t=1.0, dims=(-2, -1)):
#         super().__init__()
#         self.t = t
#         self.dims = dims

#     def forward(self, x):
#         return frequency_based_relu(x, self.t, self.dims)

In [ ]:
def freq_mag_softshrink(x, lam=1e-2, dims=(-2, -1), eps=1e-8):
    z = torch.fft.fftn(x, dim=dims)
    mag = torch.abs(z)
    scale = torch.clamp(mag - lam, min=0.0) / (mag + eps)
    z2 = z * scale
    y = torch.fft.ifftn(z2, dim=dims)
    return y.real

class fReLU(nn.Module):
    def __init__(self, lam=1e-2, dims=(-2, -1), learnable=False):
        super().__init__()
        self.dims = dims
        if learnable:
            self.lam = nn.Parameter(torch.tensor(float(lam)))
        else:
            self.register_buffer("lam", torch.tensor(float(lam)))

    def forward(self, x):
        lam = torch.relu(self.lam)
        return freq_mag_softshrink(x, lam=lam, dims=self.dims)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, kernel_size, stride, padding, activation):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=kernel_size, stride=stride, padding=padding),
            activation,
            nn.Conv2d(hidden_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
            activation,
        )

    def forward(self, x):
        return self.conv(x)

In [ ]:
class Net_with_rizz(nn.Module):
    def __init__(self):
        super().__init__()
        activation = nn.ReLU()
        self.conv_block_1 = ConvBlock(3, 32, 64, 3, 1, 1, activation)
        self.conv_block_2 = ConvBlock(64, 64, 64, 3, 1, 1, activation)
        self.conv_block_3 = ConvBlock(64, 128, 256, 3, 1, 1, activation)
        self.freq_based_activation = fReLU(learnable=True)
        self.pool = nn.MaxPool2d(2)

        self.classification_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        kek = self.pool(x) # 16x16
        # (batch_size, 64, 16, 16)
        x = self.conv_block_2(kek) + self.freq_based_activation(kek)
        x = self.pool(x) # 8x8
        # (batch_size, 64, 8, 8)
        x = self.conv_block_3(x)
        x = self.classification_head(x)
        return x

In [ ]:
class Models:
    def __init__(self, model=None):
        self.model = model

    def init_model(self, activation_fn=None, optimizer=None, loss_fn=None):
        if activation_fn is None:
            activation_fn = nn.ReLU()

        if loss_fn is None:
            loss_fn = nn.CrossEntropyLoss()

        if self.model:
            print("Create additional class!")
        else:
            self.model = Net(activation_fn)

        self.loss_fn = loss_fn

        if optimizer is None:
            optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)

        self.optimizer = optimizer

    def init_data(self, train_dataset=None, test_dataset=None, transformations=None):
        if transformations is None:
            transformations = v2.Compose([
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
            ])
        self.transform = transformations

        if train_dataset is None or test_dataset is None:
            train_dataset = datasets.CIFAR10(root='./data',
                                                train=True,
                                                download=True,
                                                transform=self.transform)

            test_dataset = datasets.CIFAR10(root='./data',
                                                train=False,
                                                download=True,
                                                transform=self.transform)

        self.train_dataset = train_dataset
        self.test_dataset = test_dataset

    def train(self, epochs=10, lr=1e-3, batch_size=128):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(device)

        train_loader, val_loader = prepare_data(batch_size,
                                                self.transform)

        train(self.model, train_loader, val_loader, epochs, self.optimizer, self.loss_fn, device)

        torch.save(self.model.state_dict(), "./models")

    def check(self):
        return summary(self.model, (3, 32, 32))

In [ ]:
def validate(model, val_loader, criterion, device):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    for inputs, labels in val_loader:

        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.long()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        val_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        val_total += labels.size(0)
        val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader)

    return val_loss, val_correct / val_total

In [ ]:
def train(model, train_loader, val_loader, epochs, optimizer, criterion, device):

    for epoch in range(epochs):

        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for inputs, labels in train_loader:

            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.long()

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_loss /= len(train_loader)
        train_acc = train_correct / train_total

        val_loss, val_acc = validate(model, val_loader, criterion, device)
        print(f'Epoch: {epoch+1}, train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f}')

In [ ]:
def prepare_data(batch_size, TRANSFORMS):

    if TRANSFORMS is None:
        TRANSFORMS = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ])

    train_cifar10 = datasets.CIFAR10(root='./data',
                                                train=True,
                                                download=True,
                                                transform=TRANSFORMS)

    test_cifar10 = datasets.CIFAR10(root='./data',
                                                train=False,
                                                download=True,
                                                transform=TRANSFORMS)

    train_loader = torch.utils.data.DataLoader(train_cifar10,
                                            batch_size=batch_size,
                                            shuffle=True,
                                            num_workers=2)

    test_loader = torch.utils.data.DataLoader(test_cifar10,
                                            batch_size=batch_size,
                                            shuffle=False,
                                            num_workers=2)

    return train_loader, test_loader

def prepare_model(activation, lr):

    model = Net(activation)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()

    return model, optimizer, loss

In [ ]:
m = Models()
m.init_model()

In [ ]:
m.init_data()

In [ ]:
m.train(epochs=10)

Epoch: 1, train loss: 1.8348, train acc: 0.3055, val loss: 1.5115, val acc: 0.4331
Epoch: 2, train loss: 1.4054, train acc: 0.4777, val loss: 1.2296, val acc: 0.5456
Epoch: 3, train loss: 1.1827, train acc: 0.5683, val loss: 1.1014, val acc: 0.5983
Epoch: 4, train loss: 1.0210, train acc: 0.6324, val loss: 0.9873, val acc: 0.6481
Epoch: 5, train loss: 0.9150, train acc: 0.6739, val loss: 0.9034, val acc: 0.6763
Epoch: 6, train loss: 0.8300, train acc: 0.7026, val loss: 0.8170, val acc: 0.7154
Epoch: 7, train loss: 0.7612, train acc: 0.7313, val loss: 0.8032, val acc: 0.7134
Epoch: 8, train loss: 0.7048, train acc: 0.7522, val loss: 0.7442, val acc: 0.7357
Epoch: 9, train loss: 0.6518, train acc: 0.7712, val loss: 0.7446, val acc: 0.7407
Epoch: 10, train loss: 0.5974, train acc: 0.7920, val loss: 0.6913, val acc: 0.7616


In [ ]:
frelu = Models()
frelu.init_model(activation_fn=fReLU())

In [ ]:
frelu.init_data()

In [ ]:
frelu.train(epochs=10)

Epoch: 1, train loss: 1.7423, train acc: 0.3563, val loss: 1.4829, val acc: 0.4448
Epoch: 2, train loss: 1.2864, train acc: 0.5436, val loss: 1.2090, val acc: 0.5702
Epoch: 3, train loss: 1.0840, train acc: 0.6215, val loss: 1.0114, val acc: 0.6506
Epoch: 4, train loss: 0.9560, train acc: 0.6670, val loss: 0.9547, val acc: 0.6676
Epoch: 5, train loss: 0.8589, train acc: 0.7028, val loss: 0.8757, val acc: 0.7043
Epoch: 6, train loss: 0.7840, train acc: 0.7296, val loss: 0.8203, val acc: 0.7172
Epoch: 7, train loss: 0.7156, train acc: 0.7537, val loss: 0.8139, val acc: 0.7278
Epoch: 8, train loss: 0.6626, train acc: 0.7711, val loss: 0.8370, val acc: 0.7199
Epoch: 9, train loss: 0.6022, train acc: 0.7908, val loss: 0.7948, val acc: 0.7363
Epoch: 10, train loss: 0.5462, train acc: 0.8100, val loss: 0.8166, val acc: 0.7322


In [ ]:
frelu_not_trainable = Models()
frelu_not_trainable.init_model(activation_fn=fReLU(learnable=True))

In [ ]:
frelu_not_trainable.init_data()

In [ ]:
frelu_not_trainable.train(epochs=10)

Epoch: 1, train loss: 1.7419, train acc: 0.3554, val loss: 1.4128, val acc: 0.4956
Epoch: 2, train loss: 1.2731, train acc: 0.5498, val loss: 1.1856, val acc: 0.5881
Epoch: 3, train loss: 1.0868, train acc: 0.6221, val loss: 1.0743, val acc: 0.6269
Epoch: 4, train loss: 0.9530, train acc: 0.6729, val loss: 0.9160, val acc: 0.6813
Epoch: 5, train loss: 0.8620, train acc: 0.7032, val loss: 0.8895, val acc: 0.6968
Epoch: 6, train loss: 0.7786, train acc: 0.7344, val loss: 0.9053, val acc: 0.6936
Epoch: 7, train loss: 0.7134, train acc: 0.7555, val loss: 0.8188, val acc: 0.7266
Epoch: 8, train loss: 0.6571, train acc: 0.7747, val loss: 0.8530, val acc: 0.7123
Epoch: 9, train loss: 0.6005, train acc: 0.7943, val loss: 0.8270, val acc: 0.7301
Epoch: 10, train loss: 0.5522, train acc: 0.8098, val loss: 0.8393, val acc: 0.7263


### Residual + fReLU

In [ ]:
class Net_with_res(nn.Module):
    def __init__(self):
        super().__init__()
        activation = nn.ReLU()
        self.conv_block_1 = ConvBlock(3, 32, 64, 3, 1, 1, activation)
        self.conv_block_2 = ConvBlock(64, 64, 64, 3, 1, 1, activation)
        self.conv_block_3 = ConvBlock(64, 128, 256, 3, 1, 1, activation)
        self.freq_based_activation = fReLU(learnable=True)
        self.pool = nn.MaxPool2d(2)

        self.classification_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        kek = self.pool(x) # 16x16
        # (batch_size, 64, 16, 16)
        x = self.conv_block_2(kek) + self.freq_based_activation(kek)
        x = self.pool(x) # 8x8
        # (batch_size, 64, 8, 8)
        x = self.conv_block_3(x)
        x = self.classification_head(x)
        return x

In [ ]:
wtf = Models(Net_with_res())
wtf.init_model()
wtf.init_data()
wtf.train(epochs=20)

Create additional class!
Epoch: 1, train loss: 1.8089, train acc: 0.3125, val loss: 1.4909, val acc: 0.4502
Epoch: 2, train loss: 1.3756, train acc: 0.4925, val loss: 1.2948, val acc: 0.5223
Epoch: 3, train loss: 1.1570, train acc: 0.5813, val loss: 1.0910, val acc: 0.6017
Epoch: 4, train loss: 1.0104, train acc: 0.6354, val loss: 1.0091, val acc: 0.6450
Epoch: 5, train loss: 0.9060, train acc: 0.6780, val loss: 0.9094, val acc: 0.6774
Epoch: 6, train loss: 0.8359, train acc: 0.7036, val loss: 0.8707, val acc: 0.6872
Epoch: 7, train loss: 0.7711, train acc: 0.7284, val loss: 0.8312, val acc: 0.7070
Epoch: 8, train loss: 0.7144, train acc: 0.7489, val loss: 0.8269, val acc: 0.7174
Epoch: 9, train loss: 0.6622, train acc: 0.7661, val loss: 0.7046, val acc: 0.7575
Epoch: 10, train loss: 0.6090, train acc: 0.7853, val loss: 0.7073, val acc: 0.7589
Epoch: 11, train loss: 0.5639, train acc: 0.8032, val loss: 0.6677, val acc: 0.7685
Epoch: 12, train loss: 0.5184, train acc: 0.8198, val loss: 

### Basic + Residual block

In [ ]:
class Net_with_res(nn.Module):
    def __init__(self):
        super().__init__()
        activation = nn.ReLU()
        self.conv_block_1 = ConvBlock(3, 32, 64, 3, 1, 1, activation)
        self.conv_block_2 = ConvBlock(64, 64, 64, 3, 1, 1, activation)
        self.conv_block_3 = ConvBlock(64, 128, 256, 3, 1, 1, activation)
        self.pool = nn.MaxPool2d(2)

        self.classification_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        kek = self.pool(x) # 16x16
        # (batch_size, 64, 16, 16)
        x = self.conv_block_2(kek) + kek
        x = self.pool(x) # 8x8
        # (batch_size, 64, 8, 8)
        x = self.conv_block_3(x)
        x = self.classification_head(x)
        return x

In [ ]:
b1b12 = Models(Net_with_res())
b1b12.init_model()
b1b12.init_data()
b1b12.train(epochs=20)

Create additional class!


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e8b8c5fee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7e8b8c5fee80>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    ^self._shutdown_workers()^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    if w.is_alive():
 ^  ^   ^ ^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^ ^^^ ^
   File "/usr/lib/py

Epoch: 1, train loss: 1.8159, train acc: 0.3114, val loss: 1.4908, val acc: 0.4512


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e8b8c5fee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7e8b8c5fee80>  
 Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      self._shutdown_workers()  
    File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
 ^    ^if w.is_alive():^^
 ^ ^ ^ ^  ^ ^^^^^^^^^^^^^^^^

Epoch: 2, train loss: 1.3550, train acc: 0.4982, val loss: 1.2074, val acc: 0.5550


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e8b8c5fee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
   Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7e8b8c5fee80>  
  Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      ^self._shutdown_workers()^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^    if w.is_alive():^
^ ^ ^ ^ ^ ^ ^ ^^^^^^^^^^

... (3 outputs left out)

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e8b8c5fee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive(): 
      ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e8b8c5fee80>^^
^Traceback (most recent call last):
^^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    self._shutdown_workers()^^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^^    if w.is_alive():^^
^ 

Epoch: 5, train loss: 0.9372, train acc: 0.6615, val loss: 0.9163, val acc: 0.6751


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e8b8c5fee80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7e8b8c5fee80>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    
self._shutdown_workers()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        assert self._parent_pid == os.getpid(), 'can only test a child process'
if w.is_alive(): 
                 ^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 